# 02 · Вектор управления

Изменение поведения **без обучения**: веса не трогаются, добавляется сдвиг к активациям одного слоя.

Идея: абстрактные свойства ответа представлены в активациях приблизительно линейно. Берём два набора текстов, различающихся только нужным свойством, снимаем активации, вычитаем средние — получаем направление. Прибавляем его при генерации.

**Зачем делать первым:** минуты вместо часов. Если эффекта нет ни при каком коэффициенте, дообучение на тех же примерах скорее всего тоже не поможет — свойство не выделяется линейно.

In [ ]:
from common import MODEL_ID, SYSTEM, demo_answers, show, side_by_side, policy_suite, fmt, read_raw

import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from vlmkit import memory_report, evaluate as ev
from vlmkit.steering import SteeringVector, decoder_layers, suggest_layer

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map={"": 0},
    attn_implementation="sdpa", trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, max_pixels=1003520)
print(memory_report())

## До

Без системного промпта — чтобы эффект вектора не смешивался с эффектом инструкции.

In [ ]:
suite = policy_suite(system=None)

before = demo_answers(model, processor)
show(before, "БЕЗ ВЕКТОРА")
print("\nметрики:", fmt(ev.run(model, processor, suite)))

## Строим вектор

Контрастные пары — прямо из датасета: эталонные ответы `clarify` и `guide` против ответов `answer`. Различаются они ровно тем свойством, которое нужно: возвращает ли ответ решение студенту.

Слой — середина сети. У Qwen3.5 архитектура гибридная: часть слоёв линейное внимание, часть полное. Печатаем типы, чтобы знать, куда попали.

In [ ]:
raw = read_raw("policy.jsonl")
answers_of = lambda g: [d["answer"] for d in raw if d["group"] == g]

layers = decoder_layers(model)
layer = suggest_layer(model)
print(f"слоёв {len(layers)}, берём {layer}, тип {getattr(layers[layer], 'block_type', '?')}")
print("типы вокруг:", [(i, getattr(layers[i], "block_type", "?")[:6]) for i in range(layer - 3, layer + 4)])

vector = SteeringVector.from_contrast(
    model, processor,
    positive=answers_of("clarify") + answers_of("guide"),
    negative=answers_of("answer"),
    layer=layer,
)
print("норма направления до нормировки была скрыта; вектор единичный:", float(vector.direction.norm()))

## Развёртка по силе

Единственный параметр — `strength`. Ниже единицы эффект обычно не виден, выше 3–4 рушится связность. Отрицательные значения **подавляют** свойство.

Смотрите на обе метрики: если попадание растёт вместе с ложными — вектор сдвигает поведение целиком, а не там, где нужно.

In [ ]:
results = {}
for s in (0.5, 1.0, 2.0, 3.0):
    with vector.applied(model, strength=s):
        results[s] = ev.run(model, processor, suite)
    print(f"s={s:<4} {fmt(results[s])}")

## До/после на лучшем коэффициенте

Лучший — тот, где попадание уже высокое, а ложные ещё не поползли.

In [ ]:
best = max(results, key=lambda s: results[s]["hit"] - results[s]["false"])
print(f"лучший s = {best}")

with vector.applied(model, strength=best):
    after = demo_answers(model, processor)

show(after, f"С ВЕКТОРОМ, s={best}")
side_by_side(before, after, detector=lambda t: "?" in t)

vector.save("policy-vector.pt")

## Обратное направление

Тот же вектор со знаком минус должен **подавлять** уточнения — модель станет выдавать готовое даже там, где не должна. Это проверка, что вектор поймал именно нужное свойство, а не шум.

In [ ]:
with vector.applied(model, strength=-best):
    print(f"s={-best}:", fmt(ev.run(model, processor, suite)))

## Вывод

**Эффект есть и растёт с s** — свойство линейно, дообучение тоже сработает, и можно идти в `01-sft`.

**Эффекта нет** — прежде чем собирать данные, пересмотрите постановку: возможно, «вернуть решение студенту» не одно свойство, а несколько разных.

**Попадание и ложные растут вместе** — вектор слишком грубый для этой задачи. Он хорош как проверка достижимости и как регулятор поверх обученной модели, но не как единственный механизм.